# Curvature-Coupled Dark Energy: Weak Lensing and Late-ISW Angular Spectra

This notebook reproduces the light-cone observables shown in the
**Curvature-Coupled Dark Energy (CCDE)** paper:

- **Fig. 10:** weak-lensing convergence spectra for source planes at
  $z_s=0.85$ and $z_s=3.3$;
- **Fig. 12:** late Integrated Sachs--Wolfe angular spectra for line-of-sight
  integrals truncated at $z_s=0.85$ and $z_s=3.3$.

Only the calculations required for these paper figures are retained.


In [ ]:

import os
from os.path import exists
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, MultipleLocator, FormatStrFormatter


def nested_dict(n, value_type):
    """Create an n-level nested defaultdict."""
    if n == 1:
        return defaultdict(value_type)
    return defaultdict(lambda: nested_dict(n - 1, value_type))


data = nested_dict(7, list)

# -------------------------------------------------
# Paper-wide plotting convention
# -------------------------------------------------
text_size = 30
fig_size_x = 28
fig_size_y = 10
label_fs = 34
lw_f = 3

plt.rc('text', usetex=True)
plt.rc('font', family='normal', weight='bold', size=text_size)

major_alpha = 0.35
minor_alpha = 0.15

# Fixed model-to-colour mapping used throughout the CCDE notebooks.
colors = [
    '#000000',  # (sigma=0.001, alpha=0.001)
    '#0072B2',  # (sigma=0.3,   alpha=0.1)
    '#E69F00',  # (sigma=0.3,   alpha=0.001)
    '#009E73',  # (sigma=0.3,   alpha=-0.1)
    '#D55E00',  # (sigma=1,     alpha=0.1)
    '#56B4E9',  # (sigma=1,     alpha=-0.1)
    '#CC79A7',  # (sigma=1.5,   alpha=0.1)
    '#F0E442',  # (sigma=1.5,   alpha=0.001)
    '#882255',  # (sigma=1.5,   alpha=-0.05)
]

alpha_consts = [
    "0.001", "0.1", "0.001", "-0.1",
    "0.1", "-0.1", "0.1", "0.001", "-0.05"
]
sigma_consts = [
    "0.001", "0.3", "0.3", "0.3",
    "1.", "1.", "1.5", "1.5", "1.5"
]


## 1. Load CCDE light-cone angular spectra

The angular spectra are stored for two late-time line-of-sight configurations,
with upper limits/source planes at $z_s=0.85$ and $z_s=3.3$.

The original data construction is retained through

`data["C_el"][z_s][sigma_key][alpha_key]`.

The CLASS-format angular-spectrum files contain the temperature spectrum and two
lensing-potential auto-spectra, `lens[1]-lens[1]` and `lens[2]-lens[2]`.


In [ ]:
# -------------------------------------------------
# Configuration
# -------------------------------------------------

data_root = "./../DataGenerator/"

data_address_1 = os.path.join(data_root, "isw1")
data_address_2 = os.path.join(data_root, "isw2")

load_data = True

alphas = ["0.1", "0.05", "0.001", "-0.05", "-0.1"]
sigmas = ["0.001", "0.3", "1.", "1.5"]

z_isw1 = 0.85
z_isw2 = 3.3


# -------------------------------------------------
# CLASS angular-spectrum columns
#
# Python uses zero-based indexing.
#
# 1:l
# 2:TT
# ...
# 9:lens[1]-lens[1]
# 10:lens[2]-lens[2]
# -------------------------------------------------

CL_ELL = 0
CL_TT = 1
CL_LENS_1 = 8
CL_LENS_2 = 9


# -------------------------------------------------
# Helpers
# -------------------------------------------------

def skey(s):
    return f"sigma={s}"


def akey(a):
    return f"alpha={a}"


# -------------------------------------------------
# Stable parameter-grid indexing
# -------------------------------------------------

alpha = np.array(
    sorted(set(alphas), key=float),
    dtype=object
)

sigma = np.array(
    sorted(set(sigmas), key=float),
    dtype=object
)

alpha_to_j = {
    a: j
    for j, a in enumerate(alpha)
}

sigma_to_i = {
    s: i
    for i, s in enumerate(sigma)
}

out_arr = np.empty(
    (len(sigma), len(alpha)),
    dtype=object
)


# -------------------------------------------------
# Preserve the original data structure
#
# data['C_el'][z_source][sigma_key][alpha_key]
# -------------------------------------------------

data.setdefault('C_el', {})

data['C_el'].setdefault(
    z_isw1,
    {}
)

data['C_el'].setdefault(
    z_isw2,
    {}
)


# -------------------------------------------------
# LambdaCDM reference
# -------------------------------------------------

lcdm_sigma = "0.0"
lcdm_alpha = "0.0"

lcdm_file_1 = os.path.join(
    data_address_1,
    "LCDM",
    "LCDM_cl.dat"
)

lcdm_file_2 = os.path.join(
    data_address_2,
    "LCDM",
    "LCDM_cl.dat"
)


# Initialize LambdaCDM entries.
data['C_el'][z_isw1].setdefault(
    skey(lcdm_sigma),
    {}
)

data['C_el'][z_isw2].setdefault(
    skey(lcdm_sigma),
    {}
)

data['C_el'][z_isw1][
    skey(lcdm_sigma)
][
    akey(lcdm_alpha)
] = None

data['C_el'][z_isw2][
    skey(lcdm_sigma)
][
    akey(lcdm_alpha)
] = None


# Load LambdaCDM spectra.
if exists(lcdm_file_1):

    print(
        "\033[94mLoading LambdaCDM isw1 spectrum\033[0m"
    )

    data['C_el'][z_isw1][
        skey(lcdm_sigma)
    ][
        akey(lcdm_alpha)
    ] = np.loadtxt(
        lcdm_file_1
    )

else:

    print(
        "\033[91mMissing:\033[0m",
        lcdm_file_1
    )


if exists(lcdm_file_2):

    print(
        "\033[94mLoading LambdaCDM isw2 spectrum\033[0m"
    )

    data['C_el'][z_isw2][
        skey(lcdm_sigma)
    ][
        akey(lcdm_alpha)
    ] = np.loadtxt(
        lcdm_file_2
    )

else:

    print(
        "\033[91mMissing:\033[0m",
        lcdm_file_2
    )


# -------------------------------------------------
# CCDE parameter grid
# -------------------------------------------------

total_loaded_files = 0
missing_files = []


if load_data:

    for a_str in alpha:

        for s_str in sigma:

            output_dir = (
                f"sigma{s_str}_alpha{a_str}"
            )

            run_dir = (
                f"run_{output_dir}"
            )


            # -----------------------------------------
            # isw1
            # -----------------------------------------

            path_isw1 = os.path.join(
                data_address_1,
                run_dir,
                (
                    f"CCDE_sigma{s_str}_"
                    f"alpha{a_str}_cl.dat"
                )
            )


            # -----------------------------------------
            # isw2
            # -----------------------------------------

            path_isw2 = os.path.join(
                data_address_2,
                run_dir,
                (
                    f"CCDE_sigma{s_str}_"
                    f"alpha{a_str}_cl.dat"
                )
            )


            # -----------------------------------------
            # Store run label
            # -----------------------------------------

            i = sigma_to_i[s_str]
            j = alpha_to_j[a_str]

            out_arr[i][j] = output_dir


            # -----------------------------------------
            # Initialize dictionary entries
            # -----------------------------------------

            data['C_el'][z_isw1].setdefault(
                skey(s_str),
                {}
            )

            data['C_el'][z_isw2].setdefault(
                skey(s_str),
                {}
            )

            data['C_el'][z_isw1][
                skey(s_str)
            ][
                akey(a_str)
            ] = None

            data['C_el'][z_isw2][
                skey(s_str)
            ][
                akey(a_str)
            ] = None


            # -----------------------------------------
            # Load isw1 spectrum
            # -----------------------------------------

            if exists(path_isw1):

                data['C_el'][z_isw1][
                    skey(s_str)
                ][
                    akey(a_str)
                ] = np.loadtxt(
                    path_isw1
                )

                total_loaded_files += 1

            else:

                missing_files.append(
                    path_isw1
                )


            # -----------------------------------------
            # Load isw2 spectrum
            # -----------------------------------------

            if exists(path_isw2):

                data['C_el'][z_isw2][
                    skey(s_str)
                ][
                    akey(a_str)
                ] = np.loadtxt(
                    path_isw2
                )

                total_loaded_files += 1

            else:

                missing_files.append(
                    path_isw2
                )


# -------------------------------------------------
# Summary
# -------------------------------------------------

print(
    "Number of CCDE spectrum files loaded:",
    total_loaded_files
)

print(
    "Alphas:",
    list(alpha)
)

print(
    "Sigmas:",
    list(sigma)
)


expected_ccde_files = (
    len(alpha)
    * len(sigma)
    * 2
)

print(
    "Expected number of CCDE spectrum files:",
    expected_ccde_files
)


if missing_files:

    print(
        f"\033[93mMissing "
        f"{len(missing_files)} files "
        f"(showing up to 10):\033[0m"
    )

    for filename in missing_files[:10]:

        print(
            " -",
            filename
        )

else:

    print(
        "\033[92mAll CCDE spectrum files were found.\033[0m"
    )

## 2. Weak-lensing convergence spectra — paper Fig. 10

The two lensing-potential columns correspond to source planes at
$z_s=0.85$ and $z_s=3.3$. With
$
C_\ell^{\kappa\kappa}
=
\frac{1}{4}\ell^2(\ell+1)^2 C_\ell^{\varphi_L\varphi_L},
$
the upper panels show
$\ell(\ell+1)C_\ell^{\kappa\kappa}/(2\pi)$.

The lower panels show ratios to the corresponding $\Lambda$CDM predictions.
Both source-plane spectra are read from the same `cl.dat` file, because both
lensing windows are stored simultaneously in that output.


In [ ]:

# -------------------------------------------------
# LambdaCDM lensing baseline
# -------------------------------------------------
baseline_sigma_key = 'sigma=0.0'
baseline_alpha_key = 'alpha=0.0'

# Both source-plane lensing spectra are stored in one angular-spectrum file.
cl_base_lens = data['C_el'][z_isw1][baseline_sigma_key][baseline_alpha_key]

if not isinstance(cl_base_lens, np.ndarray):
    raise RuntimeError("LambdaCDM angular spectra are required for paper Fig. 10.")

ell_class = cl_base_lens[:, CL_ELL]
Cl_base_z085 = cl_base_lens[:, CL_LENS_1]
Cl_base_z33 = cl_base_lens[:, CL_LENS_2]

eps = 1.e-30
Cl_base_z085_safe = np.where(np.abs(Cl_base_z085) > eps, Cl_base_z085, eps)
Cl_base_z33_safe = np.where(np.abs(Cl_base_z33) > eps, Cl_base_z33, eps)

# CLASS stores l(l+1) C_l^{phi_L phi_L}/(2 pi).
# Multiplication by [l(l+1)]^2/4 gives
# l(l+1) C_l^{kappa kappa}/(2 pi).
norm_kappa = (ell_class * (ell_class + 1.0))**2 / 4.0

# -------------------------------------------------
# Figure
# -------------------------------------------------
fig = plt.figure(figsize=(fig_size_x, fig_size_y), facecolor='w')

gs = fig.add_gridspec(
    2, 2,
    height_ratios=[2.0, 1.5],
    wspace=0.00,
    hspace=0.05
)

ax_lens1 = fig.add_subplot(gs[0, 0])
ax_lens2 = fig.add_subplot(gs[0, 1], sharex=ax_lens1, sharey=ax_lens1)
ax_lens1_rat = fig.add_subplot(gs[1, 0], sharex=ax_lens1)
ax_lens2_rat = fig.add_subplot(gs[1, 1], sharex=ax_lens2, sharey=ax_lens1_rat)

all_axes = [ax_lens1, ax_lens2, ax_lens1_rat, ax_lens2_rat]

for ax in all_axes:
    ax.set_xscale('log')
    ax.tick_params(which='both', direction='in', top=True, right=True)

    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )

    ax.grid(True, which='major', alpha=major_alpha)
    ax.grid(True, which='minor', alpha=minor_alpha)

    ax.tick_params(which='major', length=8, width=1.3)
    ax.tick_params(which='minor', length=4, width=1.0)

ax_lens1.set_yscale('log')
ax_lens2.set_yscale('log')

ax_lens2.tick_params(labelleft=False)
ax_lens2_rat.tick_params(labelleft=False)

for ax in [ax_lens1, ax_lens2]:
    ax.tick_params(labelbottom=False)

# -------------------------------------------------
# Plot models
# -------------------------------------------------
for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    sigma_key = f'sigma={sigma_l}'
    alpha_key = f'alpha={alpha_l}'

    if sigma_key not in data['C_el'][z_isw1]:
        continue
    if alpha_key not in data['C_el'][z_isw1][sigma_key]:
        continue

    cl = data['C_el'][z_isw1][sigma_key][alpha_key]

    if not isinstance(cl, np.ndarray):
        continue

    ell = cl[:, CL_ELL]
    Cl_z085 = cl[:, CL_LENS_1]
    Cl_z33 = cl[:, CL_LENS_2]

    if not np.array_equal(ell, ell_class):
        raise ValueError(
            "Model and LambdaCDM ell grids differ; interpolate before taking ratios."
        )

    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'
    color = colors[num]

    ax_lens1.loglog(
        ell, Cl_z085 * norm_kappa, '-',
        lw=lw_f, c=color
    )

    ax_lens2.loglog(
        ell, Cl_z33 * norm_kappa, '-',
        lw=lw_f, c=color, label=label
    )

    ax_lens1_rat.semilogx(
        ell, Cl_z085 / Cl_base_z085_safe, '-',
        lw=lw_f, c=color
    )

    ax_lens2_rat.semilogx(
        ell, Cl_z33 / Cl_base_z33_safe, '-',
        lw=lw_f, c=color
    )

# -------------------------------------------------
# Labels and ranges
# -------------------------------------------------
ax_lens1.text(
    0.05, 0.80, r'$z_s=0.85$',
    transform=ax_lens1.transAxes, fontsize=40
)

ax_lens2.text(
    0.05, 0.80, r'$z_s=3.3$',
    transform=ax_lens2.transAxes, fontsize=40
)

ax_lens1.set_ylabel(
    r'$\frac{\ell(\ell+1)}{2\pi}C_\ell^{\kappa\kappa}$',
    fontsize=label_fs
)

ax_lens1_rat.set_ylabel(
    r'$C_\ell/C_\ell^{\Lambda{\rm CDM}}$',
    fontsize=label_fs
)

ax_lens1_rat.set_xlabel(r'$\ell$', fontsize=label_fs)
ax_lens2_rat.set_xlabel(r'$\ell$', fontsize=label_fs)

for ax in all_axes:
    ax.set_xlim(2, 1.e3)

ax_lens1.set_ylim(1.e-10, 1.e-3)
ax_lens2.set_ylim(1.e-10, 1.e-3)

ax_lens1_rat.set_ylim(0.45, 1.5)
ax_lens2_rat.set_ylim(0.45, 1.5)

for ax in [ax_lens1_rat, ax_lens2_rat]:
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.1))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

ax_lens2.legend(
    loc='upper left',
    bbox_to_anchor=(0.25, 0.61),
    frameon=False,
    fontsize=22,
    ncol=2,
    columnspacing=0.2
)

# plt.tight_layout()
os.makedirs("./Figs", exist_ok=True)

plt.savefig(
    './Figs/Cl_kappa_ratios.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()


## 3. Late-ISW angular spectra — paper Fig. 12

The two light-cone configurations truncate the line-of-sight ISW integral at
$z_s=0.85$ and $z_s=3.3$. The CLASS-format temperature column is stored as
$\ell(\ell+1)C_\ell^{\Theta\Theta}/(2\pi)$, so multiplying by one additional
factor $\ell(\ell+1)$ gives the quantity shown in the paper,
$
\frac{[\ell(\ell+1)]^2}{2\pi}C_\ell^{\Theta\Theta}.
$

The lower panels show
$|C_\ell/C_\ell^{\Lambda{\rm CDM}}-1|$.
Sharp dips occur when a model crosses the corresponding $\Lambda$CDM spectrum.


In [ ]:

# -------------------------------------------------
# LambdaCDM late-ISW baselines
# -------------------------------------------------
cl_base_isw1 = data['C_el'][z_isw1][baseline_sigma_key][baseline_alpha_key]
cl_base_isw2 = data['C_el'][z_isw2][baseline_sigma_key][baseline_alpha_key]

if not isinstance(cl_base_isw1, np.ndarray):
    raise RuntimeError("LambdaCDM z_s=0.85 spectrum is required for paper Fig. 12.")

if not isinstance(cl_base_isw2, np.ndarray):
    raise RuntimeError("LambdaCDM z_s=3.3 spectrum is required for paper Fig. 12.")

ell_class_1 = cl_base_isw1[:, CL_ELL]
ell_class_2 = cl_base_isw2[:, CL_ELL]

Cl_base_isw1 = cl_base_isw1[:, CL_TT]
Cl_base_isw2 = cl_base_isw2[:, CL_TT]

eps = 1.e-30
Cl_base_isw1_safe = np.where(np.abs(Cl_base_isw1) > eps, Cl_base_isw1, eps)
Cl_base_isw2_safe = np.where(np.abs(Cl_base_isw2) > eps, Cl_base_isw2, eps)

# -------------------------------------------------
# Figure
# -------------------------------------------------
fig = plt.figure(figsize=(fig_size_x, fig_size_y), facecolor='w')

gs = fig.add_gridspec(
    2, 2,
    height_ratios=[2.0, 1.5],
    wspace=0.00,
    hspace=0.05
)

ax_isw1 = fig.add_subplot(gs[0, 0])
ax_isw2 = fig.add_subplot(gs[0, 1], sharey=ax_isw1)
ax_isw1_rat = fig.add_subplot(gs[1, 0], sharex=ax_isw1)
ax_isw2_rat = fig.add_subplot(gs[1, 1], sharex=ax_isw2, sharey=ax_isw1_rat)

all_axes = [ax_isw1, ax_isw2, ax_isw1_rat, ax_isw2_rat]

for ax in all_axes:
    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.tick_params(which='both', direction='in', top=True, right=True)

    ax.xaxis.set_major_locator(LogLocator(base=10, numticks=20))
    ax.xaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )

    ax.yaxis.set_major_locator(LogLocator(base=10, numticks=20))
    ax.yaxis.set_minor_locator(
        LogLocator(base=10, subs=np.arange(2, 10) * 0.1)
    )

    ax.grid(True, which='major', alpha=major_alpha)
    ax.grid(True, which='minor', alpha=minor_alpha)

    ax.tick_params(which='major', length=8, width=1.3)
    ax.tick_params(which='minor', length=4, width=1.0)

ax_isw2.tick_params(labelleft=False)
ax_isw2_rat.tick_params(labelleft=False)

for ax in [ax_isw1, ax_isw2]:
    ax.tick_params(labelbottom=False)

# -------------------------------------------------
# Plot models
# -------------------------------------------------
for num, (sigma_l, alpha_l) in enumerate(zip(sigma_consts, alpha_consts)):

    sigma_key = f'sigma={sigma_l}'
    alpha_key = f'alpha={alpha_l}'

    if sigma_key not in data['C_el'][z_isw1]:
        continue
    if alpha_key not in data['C_el'][z_isw1][sigma_key]:
        continue

    cl1 = data['C_el'][z_isw1][sigma_key][alpha_key]
    cl2 = data['C_el'][z_isw2][sigma_key][alpha_key]

    if not isinstance(cl1, np.ndarray) or not isinstance(cl2, np.ndarray):
        continue

    ell1 = cl1[:, CL_ELL]
    ell2 = cl2[:, CL_ELL]

    Cl1 = cl1[:, CL_TT]
    Cl2 = cl2[:, CL_TT]

    if not np.array_equal(ell1, ell_class_1):
        raise ValueError("z_s=0.85 model and LambdaCDM ell grids differ.")

    if not np.array_equal(ell2, ell_class_2):
        raise ValueError("z_s=3.3 model and LambdaCDM ell grids differ.")

    # CLASS already stores l(l+1) C_l/(2 pi).
    norm_isw1 = ell1 * (ell1 + 1.0)
    norm_isw2 = ell2 * (ell2 + 1.0)

    label = rf'$(\sigma={float(sigma_l):g},\,\alpha={float(alpha_l):g})$'
    color = colors[num]

    ax_isw1.loglog(
        ell1, Cl1 * norm_isw1, '-',
        lw=lw_f, c=color
    )

    ax_isw2.loglog(
        ell2, Cl2 * norm_isw2, '-',
        lw=lw_f, c=color, label=label
    )

    frac1 = np.abs(Cl1 / Cl_base_isw1_safe - 1.0)
    frac2 = np.abs(Cl2 / Cl_base_isw2_safe - 1.0)

    # Exact crossings are zero and cannot be displayed on a logarithmic axis.
    frac1 = np.where(frac1 > 0.0, frac1, np.nan)
    frac2 = np.where(frac2 > 0.0, frac2, np.nan)

    ax_isw1_rat.loglog(
        ell1, frac1, '-',
        lw=lw_f, c=color
    )

    ax_isw2_rat.loglog(
        ell2, frac2, '-',
        lw=lw_f, c=color
    )

# -------------------------------------------------
# Labels and ranges
# -------------------------------------------------
ax_isw1.text(
    0.06, 0.85, r'$z_s=0.85$',
    transform=ax_isw1.transAxes, fontsize=40
)

ax_isw2.text(
    0.06, 0.85, r'$z_s=3.3$',
    transform=ax_isw2.transAxes, fontsize=40
)

ax_isw1.set_ylabel(
    r'$\frac{[\ell(\ell+1)]^2}{2\pi}C_\ell^{\Theta\Theta}$',
    fontsize=label_fs
)

ax_isw1_rat.set_ylabel(
    r'$|C_\ell/C_\ell^{\Lambda{\rm CDM}}-1|$',
    fontsize=label_fs
)

ax_isw1_rat.set_xlabel(r'$\ell$', fontsize=label_fs)
ax_isw2_rat.set_xlabel(r'$\ell$', fontsize=label_fs)

for ax in all_axes:
    ax.set_xlim(2, 500)

ax_isw1.set_ylim(2.e-12, 5.e-8)
ax_isw2.set_ylim(2.e-12, 5.e-8)

ax_isw1_rat.set_ylim(1.e-4, 500)
ax_isw2_rat.set_ylim(1.e-4, 500)

ax_isw2.legend(
    loc='upper left',
    bbox_to_anchor=(0.10, 0.61),
    frameon=False,
    fontsize=22,
    ncol=2,
    columnspacing=0.2
)

# plt.tight_layout()

plt.savefig(
    './Figs/Cl_ISW_ratios.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.1
)

plt.show()
